# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Data Contract Statement:**
- **Unit of Analysis:** Exactly one row represents one unique pseudonymized content item (`content_id`), published under a specific client account (`client_id`).
- **Population Scope:** 30,000 unique URLs across 32 distinct clients.
- **Time Window Alignment:**
  - **Trailing Observation Window (90 Days):** All cumulative search metrics (`impressions_90d`, `clicks_90d`) and engagement metrics (`sessions_90d`, `pageviews_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`) cover the 90 days prior to snapshot cut-off.
  - **Prior Trend Window (Days 31–60 Back):** `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.
  - **Recent Comparative Window (Days 1–30 Back):** `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`.
  - **Prediction / Decision Point ($t_0$):** Features represent state knowable prior to recent outcome confirmation.

In [1]:
import os
import pandas as pd
import numpy as np

data_path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

# Verify grain: Unique content_id
duplicate_grain_count = df.duplicated(subset=["content_id"]).sum()
assert duplicate_grain_count == 0, f"Grain violation: {duplicate_grain_count} duplicate content_ids found!"

print(f"Contract Claim 1 Verified: Grain is exactly 1 row per content_id.")
print(f"Total Rows: {len(df):,}")
print(f"Unique Clients: {df['client_id'].nunique()}")
print(f"Content Age (Days): min = {df['content_age_days'].min()}, max = {df['content_age_days'].max()}, median = {df['content_age_days'].median():.0f}")
print(f"Freshness (Days Since Last Update): min = {df['days_since_last_update'].min()}, max = {df['days_since_last_update'].max()}")

Contract Claim 1 Verified: Grain is exactly 1 row per content_id.
Total Rows: 30,000
Unique Clients: 32
Content Age (Days): min = 90, max = 564, median = 236
Freshness (Days Since Last Update): min = 1, max = 373


## 2. Fields: feature / label / context / excluded

Every column in the dataset is strictly categorized into one of four mutually exclusive contract buckets:

### A. Features (Pre-decision signals knowable before outcome):
- **Continuous Traffic/Search:** `search_volume`, `competition`, `cpc`, `impressions_90d`, `clicks_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `avg_position`, `ctr`.
- **Continuous Engagement:** `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.
- **Continuous Content/Age:** `word_count`, `char_count`, `content_age_days`, `days_since_last_update`.
- **Categorical / Tiers:** `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `char_count_tier`, `impression_tier`, `position_tier`.

### B. Label / Target (The ground truth outcome):
- **`is_declining_label`:** Binary indicator ($1$ if `trend_direction == 'down'`, else $0$).

### C. Context (Grouping, validation splitting, and reporting only — NEVER model inputs):
- **`content_id`:** Pseudonymous page identifier (join and deduplication key).
- **`client_id`:** Pseudonymous domain/client identifier (mandatory for **client-holdout group splitting**).

### D. Excluded Fields (Strictly prohibited from feature vector):
- **`trend_direction` & `trend_pct`:** **Label source leakage.** The label is mathematically defined from these columns; using them creates a 100% circular, invalid model.
- **`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`:** **Lookahead leakage.** These constitute the evaluation outcome window from which trend direction is calculated.
- **`provider_used`, `model_used`:** **High missingness / non-generalizable metadata.** Unreliable generation metadata not intrinsic to content quality.

In [2]:
# Field classification specification
FEATURES_NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "impressions_prev_30d",
    "clicks_prev_30d", "sessions_prev_30d", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

FEATURES_CATEGORICAL = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier"
]

CONTEXT_COLS = ["content_id", "client_id"]
LABEL_SOURCE_COLS = ["trend_direction", "trend_pct"]
LOOKAHEAD_COLS = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]
EXCLUDED_COLS = ["provider_used", "model_used"]

total_accounted = (len(FEATURES_NUMERIC) + len(FEATURES_CATEGORICAL) + 
                   len(CONTEXT_COLS) + len(LABEL_SOURCE_COLS) + 
                   len(LOOKAHEAD_COLS) + len(EXCLUDED_COLS))

print(f"Contract Field Accounting:")
print(f"  Numeric Features:      {len(FEATURES_NUMERIC)}")
print(f"  Categorical Features:  {len(FEATURES_CATEGORICAL)}")
print(f"  Context Keys:          {len(CONTEXT_COLS)}")
print(f"  Label Source (Leakage):{len(LABEL_SOURCE_COLS)}")
print(f"  Lookahead Outcome:     {len(LOOKAHEAD_COLS)}")
print(f"  Excluded Metadata:     {len(EXCLUDED_COLS)}")
print(f"  Total Accounted:       {total_accounted} / {df.shape[1]} columns in CSV")
assert total_accounted == df.shape[1], "Contract violation: Unaccounted columns found!"
print("Verified: 100% of dataset columns explicitly accounted for.")

Contract Field Accounting:
  Numeric Features:      26
  Categorical Features:  9
  Context Keys:          2
  Label Source (Leakage):2
  Lookahead Outcome:     3
  Excluded Metadata:     2
  Total Accounted:       44 / 44 columns in CSV
Verified: 100% of dataset columns explicitly accounted for.


## 3. Verify it with queries (grain, counts, missing values, windows)

We execute rigorous programmatic audits to verify every clause of the data contract.

In [3]:
# Query 1: Verify missingness structure across columns
null_series = df.isnull().sum()
null_pct = (null_series / len(df)) * 100
missing_report = pd.DataFrame({"null_count": null_series, "null_pct": null_pct})
missing_cols = missing_report[missing_report["null_count"] > 0].sort_values("null_pct", ascending=False)

print("Columns with missing values:")
display(missing_cols)

# Query 2: Test systematic missingness pattern by content_type
print("\nMissingness in 'search_volume' by content_type (proves missingness is systematic, not random):")
display(df.groupby("content_type")["search_volume"].apply(lambda s: s.isnull().mean() * 100).round(1))

print("\nMissingness in 'word_count' by content_type:")
display(df.groupby("content_type")["word_count"].apply(lambda s: s.isnull().mean() * 100).round(1))

# Query 3: Verify rate column scaling gotcha (CTR = 0.76 means 0.76%, not 76%)
print("\nRate column summary percentiles (demonstrating x100 scale):")
display(df[["ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"]].describe().round(2))

# Query 4: Verify avg_position = 0 gotcha (means 'no position data')
zero_pos_count = (df["avg_position"] == 0).sum()
print(f"\nRows with avg_position == 0 ('no ranking position recorded'): {zero_pos_count:,} ({zero_pos_count/len(df)*100:.2f}%)")

Columns with missing values:


,null_count,null_pct
provider_used,21438,71.460000
word_count_tier,7699,25.663333
char_count,7699,25.663333
word_count,7699,25.663333
char_count_tier,7699,25.663333
model_used,5733,19.110000
trend_pct,3388,11.293333
competition_level,2610,8.700000
search_volume,2468,8.226667
competition,2468,8.226667



Missingness in 'search_volume' by content_type (proves missingness is systematic, not random):


content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4
Name: search_volume, dtype: float64


Missingness in 'word_count' by content_type:


content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64


Rate column summary percentiles (demonstrating x100 scale):


,ctr,engagement_rate,scroll_rate,ai_traffic_pct
count,30000.00,30000.00,29875.00,30000.00
mean,0.51,2.53,18.21,0.77
std,3.28,8.31,29.47,7.43
min,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00
50%,0.07,0.00,5.00,0.00
75%,0.29,1.35,23.53,0.00
max,100.00,100.00,300.00,300.00



Rows with avg_position == 0 ('no ranking position recorded'): 1,205 (4.02%)


## 4. Data limits

**What this dataset CANNOT tell you:**
1. **Unbalanced Client Histories:**
   Different clients joined FlyRank at varying dates over the 17-month panel. While this 30k starter snapshot enforces a uniform trailing 90-day aggregation window, real warehouse data features variable panel depth requiring per-client date filtering (`gsc_data_start`).
2. **GSC-Only Early History & Zero GA4 Fill:**
   Rows pre-dating client GA4 onboarding lack behavioral telemetry; in full warehouse queries, GA4 fields are zero-filled with `ga4_data_available = FALSE`. In this starter slice, zero values in `sessions_90d` or `scroll_events_90d` can reflect either true lack of visitor engagement or uninstrumented tracking.
3. **Absence of Search Page Context (SERP Features):**
   The dataset does not record whether search impression drops were caused by Google deploying AI Overviews, Local Packs, or zero-click answer boxes for the target query.
4. **No Semantic Article Text:**
   Only character and word counts are available. Keyword stuffing, semantic readability, and factual accuracy cannot be analyzed directly from numeric metadata.
5. **Observational Limitations (No True Counterfactuals):**
   We observe natural decay and stable trajectories, but we do not observe counterfactual control groups of refreshed vs unrefreshed identical articles. Hence, causal refresh uplift cannot be asserted.

In [4]:
# Data limits verification summary
data_limits = {
    "Unbalanced client panel": f"32 clients with varying page volumes (min={df.groupby('client_id').size().min()}, max={df.groupby('client_id').size().max()})",
    "Zero analytics telemetry": f"{(df['sessions_90d'] <= 1).sum():,} pages have <= 1 session over 90 days",
    "Missing keyword metadata": f"{df['search_volume'].isnull().sum():,} pages lack search volume data",
    "Missing word count": f"{df['word_count'].isnull().sum():,} pages lack word count data",
    "Observational nature": "Observational snapshot without randomized intervention controls"
}

print("Verified Data Limits:")
for k, v in data_limits.items():
    print(f"  * {k}: {v}")

Verified Data Limits:
  * Unbalanced client panel: 32 clients with varying page volumes (min=3, max=7008)
  * Zero analytics telemetry: 4,864 pages have <= 1 session over 90 days
  * Missing keyword metadata: 2,468 pages lack search volume data
  * Missing word count: 7,699 pages lack word count data
  * Observational nature: Observational snapshot without randomized intervention controls


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.